# Create the DistilBERT LibTorch Artifact

Run this notebook inside the Triton Control code-server workspace. It exports `distilbert-base-uncased-finetuned-sst-2-english` as `distilbert_sentiment/1/model.pt` for Triton's PyTorch/LibTorch backend. This export requires a GPU.

## 1. Configure the notebook runtime

In [ ]:
from pathlib import Path
import os
import sys
import time


os.environ.setdefault("USER", "workspace")
os.environ.setdefault("LOGNAME", "workspace")
os.environ.setdefault("TORCHINDUCTOR_CACHE_DIR", "/tmp/torchinductor-workspace")

MODEL_ID = "distilbert-base-uncased-finetuned-sst-2-english"
MAX_LENGTH = 32
OUTPUT_PATH = Path("distilbert_sentiment/1/model.pt")

print("Python:", sys.executable)
print("Model:", MODEL_ID)
print("Output:", OUTPUT_PATH)

## 2. Install pinned Python packages

In [ ]:
%pip install --user --force-reinstall --index-url https://download.pytorch.org/whl/cu128 "torch==2.7.1" "torchvision==0.22.1" "torchaudio==2.7.1"
%pip install --user --force-reinstall "transformers==4.53.0"

Restart the notebook kernel after this install cell, then continue below.

## 3. Check Python packages and GPU

In [ ]:
import torch
import transformers


print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("torch file:", torch.__file__)
print("transformers:", transformers.__version__)
print("transformers file:", transformers.__file__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    print("CUDA arch list:", torch.cuda.get_arch_list())

if not torch.cuda.is_available():
    raise SystemExit("CUDA is required to export this GPU LibTorch example.")

## 4. Define helpers

In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer


def from_pretrained_with_retry(loader, model_id: str, label: str, attempts: int = 3):
    last_error = None
    for attempt in range(1, attempts + 1):
        print(f"Loading {label} from {model_id} (attempt {attempt}/{attempts})...")
        try:
            result = loader.from_pretrained(model_id)
            print(f"Loaded {label}")
            return result
        except (OSError, RuntimeError) as exc:
            last_error = exc
            if attempt == attempts:
                break
            wait_seconds = attempt * 2
            print(f"Could not load {label}, retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)
    raise RuntimeError(
        f"Could not load {label} from {model_id}. Check workspace DNS/internet access "
        "to huggingface.co, or pre-populate the HuggingFace cache before running this notebook."
    ) from last_error

In [ ]:
class SentimentWrapper(torch.nn.Module):
    def __init__(self, model: torch.nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return output.logits

## 5. Download and load the HuggingFace files

In [ ]:
config = from_pretrained_with_retry(AutoConfig, MODEL_ID, "model config")
print("Model type:", config.model_type)

In [ ]:
tokenizer = from_pretrained_with_retry(AutoTokenizer, MODEL_ID, "tokenizer")

In [ ]:
base_model = from_pretrained_with_retry(
    AutoModelForSequenceClassification,
    MODEL_ID,
    "sequence classification model",
)
base_model.eval()
print("Model loaded on CPU")

## 6. Move the model to CUDA

In [ ]:
device = torch.device("cuda")
base_model = base_model.to(device)
base_model.eval()
print("Model moved to CUDA")

## 7. Trace the TorchScript model

In [ ]:
encoded = tokenizer(
    "This product is genuinely useful and easy to recommend.",
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

print("input_ids shape:", tuple(encoded["input_ids"].shape))
print("attention_mask shape:", tuple(encoded["attention_mask"].shape))

In [ ]:
model = SentimentWrapper(base_model).to(device)
model.eval()

with torch.no_grad():
    traced_model = torch.jit.trace(
        model,
        (
            encoded["input_ids"].to(device=device, dtype=torch.long),
            encoded["attention_mask"].to(device=device, dtype=torch.long),
        ),
        strict=False,
    )

print("TorchScript trace created")

## 8. Save the Triton artifact

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
traced_model.save(str(OUTPUT_PATH))
print(f"Saved {OUTPUT_PATH}")